In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector


from braket.tracking import Tracker
from qiskit_braket_provider import BraketLocalBackend, BraketProvider, to_braket


import matplotlib.pyplot as plt

In [2]:
RUN_MODE = "LOCAL"

# If REMOTE, pick a backend name printed in Cell 1:
# Examples: "SV1", "DM1", "TN1", "rigetti_Ankaa-3", "ionq_Aria-1", ...
REMOTE_BACKEND_NAME = "SV1"  # start with SV1, then switch to a QPU name when ready

print("RUN_MODE =", RUN_MODE)
print("REMOTE_BACKEND_NAME =", REMOTE_BACKEND_NAME)

RUN_MODE = LOCAL
REMOTE_BACKEND_NAME = SV1


In [3]:
from qiskit.transpiler.passes import RemoveBarriers
from qiskit.transpiler import PassManager

def strip_barriers(qc: QuantumCircuit) -> QuantumCircuit:
    pm = PassManager(RemoveBarriers())
    return pm.run(qc)

In [4]:
def build_adder_less_than_10(a_val: int, b_val: int) -> QuantumCircuit:
    """
    9-qubit in-place 4-bit ripple-carry adder.
    Inputs restricted to 0..9 (less than 10).
    
    Qubit layout (little-endian):
      A0..A3 = q0..q3
      B0..B3 = q4..q7
      Carry  = q8  (starts 0, ends as carry-out)

    Output:
      B holds low 4 bits of (a+b)
      Carry holds the 5th bit
      We measure 5 classical bits: c0..c3 = B0..B3, c4 = Carry
    """
    if not (0 <= a_val <= 9 and 0 <= b_val <= 9):
        raise ValueError("Inputs must be integers 0..9 (less than 10).")

    qc = QuantumCircuit(9, 5)

    # Encode A into q0..q3
    for i in range(4):
        if (a_val >> i) & 1:
            qc.x(i)

    # Encode B into q4..q7
    for i in range(4):
        if (b_val >> i) & 1:
            qc.x(4 + i)

    qc.barrier(label="encoded a,b")

    # Forward pass (majority)
    # majority(a,b,c): cx(c,b), cx(c,a), ccx(a,b,c)
    for i in range(4):
        a = i
        b = 4 + i
        c = 8
        qc.cx(c, b)
        qc.cx(c, a)
        qc.ccx(a, b, c)
        qc.barrier()

    # Backward pass (unmajority)
    # unmajority(a,b,c): ccx(a,b,c), cx(c,a), cx(a,b)
    for i in reversed(range(4)):
        a = i
        b = 4 + i
        c = 8
        qc.ccx(a, b, c)
        qc.cx(c, a)
        qc.cx(a, b)
        qc.barrier()

    # Measure: B0..B3 -> c0..c3, Carry -> c4
    for i in range(4):
        qc.measure(4 + i, i)
    qc.measure(8, 4)

    return qc

In [5]:
def run_adder_less_than_10(a: int, b: int, shots: int = 1000, backend_name: str = "braket_sv", show_circuit: bool = True):
    """
    Returns: (measured_sum, counts, most_likely_bitstring)

    IMPORTANT:
    - If RUN_MODE == "LOCAL": backend_name is the local simulator name ("braket_sv" or "braket_dm")
    - If RUN_MODE == "REMOTE": backend_name is ignored; we use REMOTE_BACKEND_NAME (SV1/DM1/TN1 or a QPU like rigetti_Ankaa-3)
    """
    qc = build_adder_less_than_10(a, b)

    if RUN_MODE == "REMOTE":
        qc = strip_barriers(qc)

    if show_circuit:
        print(qc.draw(output="text"))

    # -------- ONLY CHANGE: choose local vs remote backend --------
    if RUN_MODE == "LOCAL":
        backend = BraketLocalBackend("braket_sv")  # your original line
        result = backend.run(qc, shots=shots).result()
        counts = result.get_counts()
    elif RUN_MODE == "REMOTE":
        provider = BraketProvider()
        backend = provider.get_backend(REMOTE_BACKEND_NAME)
        result = backend.run(qc, shots=shots).result()
        counts = result.get_counts()
    else:
        raise ValueError("RUN_MODE must be 'LOCAL' or 'REMOTE'")
    # ------------------------------------------------------------

    top = max(counts, key=counts.get)

    # Your original decoding logic (UNCHANGED)
    carry = int(top[0])
    low4 = int(top[1:], 2)
    measured_sum = (carry << 4) | low4

    return measured_sum, counts, top


In [7]:
tracker = Tracker().start()


print("Tracker started (tracks QPU tasks).")

# -------- USER INPUT --------
a = 1
b = 6
# ----------------------------

measured_sum, counts, top = run_adder_less_than_10(
    a, b,
    shots=100,
    backend_name="SV1",   # used only when RUN_MODE="LOCAL"
    show_circuit=True
)

print("\n--- Results ---")
print("a =", a, "b =", b)
print("Most likely (c4c3c2c1c0) =", top)
print("Measured sum =", measured_sum)
print("Expected sum =", a + b)
print("Counts =", counts)


tracker.stop()

try:
    print("Estimated QPU cost (USD):", float(tracker.qpu_tasks_cost()))
except Exception as e:
    print("No QPU cost recorded (you likely ran LOCAL or only simulators).")
    print("Tracker error/info:", e)


Tracker started (tracks QPU tasks).
     ┌───┐ encoded a,b      ┌───┐      ░                 ░                 ░ »
q_0: ┤ X ├──────░───────────┤ X ├──■───░─────────────────░─────────────────░─»
     └───┘      ░           └─┬─┘  │   ░      ┌───┐      ░                 ░ »
q_1: ───────────░─────────────┼────┼───░──────┤ X ├──■───░─────────────────░─»
                ░             │    │   ░      └─┬─┘  │   ░      ┌───┐      ░ »
q_2: ───────────░─────────────┼────┼───░────────┼────┼───░──────┤ X ├──■───░─»
                ░             │    │   ░        │    │   ░      └─┬─┘  │   ░ »
q_3: ───────────░─────────────┼────┼───░────────┼────┼───░────────┼────┼───░─»
                ░      ┌───┐  │    │   ░        │    │   ░        │    │   ░ »
q_4: ───────────░──────┤ X ├──┼────■───░────────┼────┼───░────────┼────┼───░─»
     ┌───┐      ░      └─┬─┘  │    │   ░ ┌───┐  │    │   ░        │    │   ░ »
q_5: ┤ X ├──────░────────┼────┼────┼───░─┤ X ├──┼────■───░────────┼────┼───░─»
     ├───┤      